In [80]:
import pandas as pd
import numpy as np
import re

---

In [81]:
df = pd.read_csv("../../datasets/raw/steam_games_requirements.csv")
df2 = pd.read_csv("../../datasets/raw/steam_requirements_scraped.csv")

---

In [82]:
df.head(1)

,Unnamed: 0,url,types,name,desc_snippet,recent_reviews,all_reviews,release_date,developer,publisher,...,game_details,languages,achievements,genre,game_description,mature_content,minimum_requirements,recommended_requirements,original_price,discount_price
0,0,https://store.steampowered.com/app/379720/DOOM/,app,DOOM,Now includes all three premium DLC packs (Unto...,"Very Positive,(554),- 89% of the 554 user revi...","Very Positive,(42,550),- 92% of the 42,550 use...","May 12, 2016",id Software,"Bethesda Softworks,Bethesda Softworks",...,"Single-player,Multi-player,Co-op,Steam Achieve...","English,French,Italian,German,Spanish - Spain,...",54.0,Action,"About This Game Developed by id software, the...",NaN,"Minimum:,OS:,Windows 7/8.1/10 (64-bit versions...","Recommended:,OS:,Windows 7/8.1/10 (64-bit vers...",$19.99,$14.99


In [83]:
df = df[df['types'] == 'app']
df.drop(columns=['types'], inplace=True)

In [84]:
df['url'] = df['url'].apply(lambda row: row.split('/')[4])
df.rename(columns={'url': 'app_id'}, inplace=True)

In [85]:
df = df[['app_id', 'name', 'minimum_requirements', 'recommended_requirements']]

In [86]:
df.head(1)

,app_id,name,minimum_requirements,recommended_requirements
0,379720,DOOM,"Minimum:,OS:,Windows 7/8.1/10 (64-bit versions...","Recommended:,OS:,Windows 7/8.1/10 (64-bit vers..."


In [87]:
df['app_id'] = df['app_id'].astype('int64')

In [88]:
df.isna().sum()

app_id                          0
name                           14
minimum_requirements        16952
recommended_requirements    16946
dtype: int64

In [89]:
df.dropna(how='all', inplace=True)

In [90]:
df = df[~df['name'].str.contains(r'\b(OST|Soundtrack)\b')]

C:\Users\wastedy\AppData\Local\Temp\ipykernel_13828\3679353767.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df = df[~df['name'].str.contains(r'\b(OST|Soundtrack)\b')]


In [91]:
df[df[['minimum_requirements', 'recommended_requirements']].isna().all(axis=1)] # Missing

,app_id,name,minimum_requirements,recommended_requirements
16,597170,Clone Drone in the Danger Zone,NaN,NaN
20,42700,Call of Duty®: Black Ops,NaN,NaN
24,12210,Grand Theft Auto IV,NaN,NaN
26,400,Portal,NaN,NaN
28,704450,Neverwinter Nights: Enhanced Edition,NaN,NaN
...,...,...,...,...
40813,912210,Achievement Collector: Cat,NaN,NaN
40815,912140,SpaceBall in Cube,NaN,NaN
40824,906470,Gravia,NaN,NaN
40826,906430,Alive,NaN,NaN


---

In [92]:
# merge with scraped dataset
df2

,steam_appid,name,pc_requirements_minimum,pc_requirements_recommended
0,597170,Clone Drone in the Danger Zone,"<strong>Minimum:</strong><br><ul class=""bb_ul""...",NaN
1,42700,Call of Duty®: Black Ops,"<ul class=""bb_ul""><li><strong>OS *:</strong> W...",NaN
2,12210,Grand Theft Auto IV: The Complete Edition,"<ul class=""bb_ul""><li><strong>OS:</strong> Win...",NaN
3,400,Portal,<p><strong>Minimum: </strong>1.7 GHz Processor...,NaN
4,704450,Neverwinter Nights: Enhanced Edition,"<strong>Minimum:</strong><br><ul class=""bb_ul""...","<strong>Recommended:</strong><br><ul class=""bb..."
...,...,...,...,...
16185,912210,Achievement Collector: Cat,"<strong>Minimum:</strong><br><ul class=""bb_ul""...",NaN
16186,912140,SpaceBall in Cube,"<strong>Minimum:</strong><br><ul class=""bb_ul""...",NaN
16187,906470,Gravia,"<strong>Minimum:</strong><br><ul class=""bb_ul""...",NaN
16188,906430,Alive,"<strong>Minimum:</strong><br><ul class=""bb_ul""...",NaN


In [93]:
def clean_tags(text):
    if pd.isna(text):
        return None
    if '<br>' in text:
        text = re.sub('<br>', ',', text)
    if '\n' in text:
        text = re.sub('\n', ',', text)
    return " ".join(re.sub(r'<.*?>', ' ', text).split())

In [94]:
df2.dtypes

steam_appid                    int64
name                             str
pc_requirements_minimum          str
pc_requirements_recommended      str
dtype: object

In [95]:
df2.isna().sum()

steam_appid                        0
name                               8
pc_requirements_minimum          593
pc_requirements_recommended    15372
dtype: int64

In [96]:
df2.dropna(subset=['name', 'pc_requirements_minimum'], inplace=True) # The games here without names are irrelevant DLCs, games without minimum requirements are OSTs.

In [97]:
df2 = df2[~df2['name'].str.contains(r'\b(OST|Soundtrack)\b')] # Cleaning OSTs by string name

C:\Users\wastedy\AppData\Local\Temp\ipykernel_13828\3624869109.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df2 = df2[~df2['name'].str.contains(r'\b(OST|Soundtrack)\b')] # Cleaning OSTs by string name


In [98]:
df2.isna().sum()

steam_appid                        0
name                               0
pc_requirements_minimum            0
pc_requirements_recommended    13785
dtype: int64

In [99]:
df2['min_req'] = df2['pc_requirements_minimum'].apply(lambda x: clean_tags(x))

In [100]:
df2['rec_req'] = df2['pc_requirements_recommended'].apply(lambda x: clean_tags(x))

In [101]:
df2['min_req'][0]

'Minimum: , OS *: Windows 7 or newer, Processor: Modern quad-core (AMD FX-Series or newer, Intel Core i5 or faster), Memory: 2 GB RAM, Graphics: AMD Radeon HD 5770 or faster, Nvidia GeForce GT 640 or faster, Storage: 1 GB available space'

In [102]:
df2.drop(columns=['pc_requirements_minimum', 'pc_requirements_recommended'], inplace=True)

In [103]:
df2

,steam_appid,name,min_req,rec_req
0,597170,Clone Drone in the Danger Zone,"Minimum: , OS *: Windows 7 or newer, Processor...",NaN
1,42700,Call of Duty®: Black Ops,"OS *: Windows® Vista / XP / 7, Processor: Inte...",NaN
2,12210,Grand Theft Auto IV: The Complete Edition,"OS: Windows 10 (64-bit) , Processor: Intel Cor...",NaN
3,400,Portal,"Minimum: 1.7 GHz Processor, 512MB RAM, DirectX...",NaN
4,704450,Neverwinter Nights: Enhanced Edition,"Minimum: , Requires a 64-bit processor and ope...","Recommended: , Requires a 64-bit processor and..."
...,...,...,...,...
16185,912210,Achievement Collector: Cat,"Minimum: , OS *: Windows 7, 8, 10, Processor: ...",NaN
16186,912140,SpaceBall in Cube,"Minimum: , OS *: Windows 7, Processor: 1000 MH...",NaN
16187,906470,Gravia,"Minimum: , OS *: Windows 7, Processor: 3rd Gen...",NaN
16188,906430,Alive,"Minimum: , OS *: Windows 7 or newer, Processor...",NaN


In [104]:
merged = df.merge(df2, left_on='app_id', right_on='steam_appid', how='left', indicator=True)

In [105]:
merged.drop(merged[merged['name_x'].isna() & merged['name_y'].isna()].index, inplace=True) # Dropping where both name columns are NaN

In [106]:
merged.drop(merged[merged['name_x'].isna()].index, inplace=True)

In [107]:
merged[merged['_merge'] == 'both']

,app_id,name_x,minimum_requirements,recommended_requirements,steam_appid,name_y,min_req,rec_req,_merge
14,597170,Clone Drone in the Danger Zone,NaN,NaN,597170.0,Clone Drone in the Danger Zone,"Minimum: , OS *: Windows 7 or newer, Processor...",NaN,both
18,42700,Call of Duty®: Black Ops,NaN,NaN,42700.0,Call of Duty®: Black Ops,"OS *: Windows® Vista / XP / 7, Processor: Inte...",NaN,both
22,12210,Grand Theft Auto IV,NaN,NaN,12210.0,Grand Theft Auto IV: The Complete Edition,"OS: Windows 10 (64-bit) , Processor: Intel Cor...",NaN,both
23,400,Portal,NaN,NaN,400.0,Portal,"Minimum: 1.7 GHz Processor, 512MB RAM, DirectX...",NaN,both
25,704450,Neverwinter Nights: Enhanced Edition,NaN,NaN,704450.0,Neverwinter Nights: Enhanced Edition,"Minimum: , Requires a 64-bit processor and ope...","Recommended: , Requires a 64-bit processor and...",both
...,...,...,...,...,...,...,...,...,...
36102,912210,Achievement Collector: Cat,NaN,NaN,912210.0,Achievement Collector: Cat,"Minimum: , OS *: Windows 7, 8, 10, Processor: ...",NaN,both
36104,912140,SpaceBall in Cube,NaN,NaN,912140.0,SpaceBall in Cube,"Minimum: , OS *: Windows 7, Processor: 1000 MH...",NaN,both
36113,906470,Gravia,NaN,NaN,906470.0,Gravia,"Minimum: , OS *: Windows 7, Processor: 3rd Gen...",NaN,both
36115,906430,Alive,NaN,NaN,906430.0,Alive,"Minimum: , OS *: Windows 7 or newer, Processor...",NaN,both


In [108]:
merged.drop(columns=['steam_appid', 'name_y'], inplace=True)

In [109]:
merged

,app_id,name_x,minimum_requirements,recommended_requirements,min_req,rec_req,_merge
0,379720,DOOM,"Minimum:,OS:,Windows 7/8.1/10 (64-bit versions...","Recommended:,OS:,Windows 7/8.1/10 (64-bit vers...",NaN,NaN,left_only
1,578080,PLAYERUNKNOWN'S BATTLEGROUNDS,"Minimum:,Requires a 64-bit processor and opera...","Recommended:,Requires a 64-bit processor and o...",NaN,NaN,left_only
2,637090,BATTLETECH,"Minimum:,Requires a 64-bit processor and opera...","Recommended:,Requires a 64-bit processor and o...",NaN,NaN,left_only
3,221100,DayZ,"Minimum:,OS:,Windows 7/8.1 64-bit,Processor:,I...","Recommended:,OS:,Windows 10 64-bit,Processor:,...",NaN,NaN,left_only
4,8500,EVE Online,"Minimum:,OS:,Windows 7,Processor:,Intel Dual C...","Recommended:,OS:,Windows 10,Processor:,Intel i...",NaN,NaN,left_only
...,...,...,...,...,...,...,...
36117,899836,Rocksmith® 2014 Edition – Remastered – Sabaton...,"Minimum:,OS:,Windows Vista, Windows 7, Windows...","Recommended:,OS:,Windows Vista, Windows 7, Win...",NaN,NaN,left_only
36118,899832,Rocksmith® 2014 Edition – Remastered – Stone T...,"Minimum:,OS:,Windows Vista, Windows 7, Windows...","Recommended:,OS:,Windows Vista, Windows 7, Win...",NaN,NaN,left_only
36119,906840,Fantasy Grounds - Quests of Doom 4: A Midnight...,"Minimum:,OS:,Windows 7x , 8x or 10x,Processor:...","Recommended:,OS:,Windows 7x , 8x or 10x,Proces...",NaN,NaN,left_only
36120,906635,Mega Man X5 Sound Collection,"Minimum:,OS:,WINDOWS® 7 (64bit),Processor:,Int...","Recommended:,OS:,WINDOWS®10 (64bit),Processor:...",NaN,NaN,left_only


In [110]:
merged['minimum_requirements'] = merged['minimum_requirements'].fillna(merged['min_req'])

In [111]:
merged['recommended_requirements'] = merged['recommended_requirements'].fillna(merged['rec_req'])

In [112]:
merged.dropna(subset=['minimum_requirements', 'recommended_requirements', 'min_req', 'rec_req'], how='all', inplace=True)

In [113]:
merged.drop(columns=['min_req', 'rec_req', '_merge'], inplace=True)

In [114]:
merged.columns = ['appid', 'name', 'min_req', 'rec_req']

In [115]:
merged[merged.duplicated()]

,appid,name,min_req,rec_req
15044,200260,Batman: Arkham City - Game of the Year Edition,"OS *: Windows XP, Vista, 7, Processor: Intel C...",NaN


In [116]:
merged.drop_duplicates(inplace=True, ignore_index=True)

In [117]:
merged.info()

<class 'pandas.DataFrame'>
RangeIndex: 35104 entries, 0 to 35103
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   appid    35104 non-null  int64
 1   name     35104 non-null  str  
 2   min_req  35103 non-null  str  
 3   rec_req  21324 non-null  str  
dtypes: int64(1), str(3)
memory usage: 17.5 MB


In [118]:
#merged.to_csv('../../datasets/processed/mergedDF.csv')